# Assignment

Implement the following Neural Architecture Search methods using this framework:

* Random Search
* Random Sample
* Bayesian Optimisation (put some thoughts how to represent information for the surrogate
model and how to define an acquition function)
* Succesive Halving
* An iteration-based learning curve method (see, e.g., the Power-law version of [1]).

Compare the performance of these methods using some reasonable standards (e.g., give
them the same budget) on various pre-computed datasets.

Setting up NAS system

In [11]:
import os
import csv
import random
import time
import json
import pickle

import numpy as np
import matplotlib.pyplot as plt

from nats_bench import create

SEARCH_SPACES = {'tss': 'NATS-Bench topology (TSS)'}
SEARCH_SPACE = 'tss'

FAST_MODE = True
BENCH_PATH = "data/NATS-tss-v1_0-3ffb9-simple"

DATASETS = ['cifar10', 'cifar100', 'ImageNet16-120']


if not os.path.exists(BENCH_PATH):
    raise FileNotFoundError('Benchmark not found at {}. See section 1.'.format(BENCH_PATH))

api = create(BENCH_PATH, SEARCH_SPACE, fast_mode=FAST_MODE, verbose=False)

print('Search space: %s' % SEARCH_SPACES[SEARCH_SPACE])
print('Number of architectures:', len(api))

Search space: NATS-Bench topology (TSS)
Number of architectures: 15625


# Algorithms

## Random Search

In [12]:
def random_search(seed, hp_schedule, n_evals, dataset):
        records = []
        used_idx = set()
        rng = np.random.default_rng(seed)
        best_val_accuracy = -float("inf")
        best_test_accuracy = 0
        best_idx = None
        cumulative_cost = 0.0
    
    
        if dataset == 'cifar10':
            name = 'cifar10-valid'
        else:
            name = dataset

        for i in range(n_evals):
            idx = int(rng.integers(0, len(api)))

            # without replacement so keep doing it until its one that hasnt been tested before
            while idx in used_idx:
                idx = int(rng.integers(0, len(api)))

            used_idx.add(idx) 
            
            info = api.get_more_info(int(idx), name, hp=hp_schedule, is_random=False)
            val_accuracy = info["valid-accuracy"]
            test_accuracy = info["test-accuracy"]
            if best_val_accuracy < val_accuracy:
                best_val_accuracy = val_accuracy
                best_test_accuracy = test_accuracy
                best_idx = idx

            cumulative_cost += info["train-all-time"]
            
            records.append({
                "idx": int(idx),
                "best_idx": best_idx,
                "evaluation": i + 1,
                "val_accuracy": val_accuracy,
                "test_accuracy": test_accuracy,
                "best_val_accuracy": best_val_accuracy,
                "best_test_accuracy": best_test_accuracy,
                "training_cost": info["train-all-time"], # the cost of evaluating this particular architecture.
                "computation_cost": cumulative_cost,
                "epochs_trained": hp_schedule # amount of epochs trained
            })
            
        return records


## Random Sampling

In [13]:
def random_sampling(seed, hp_schedule, n_evals, dataset):
    """
    Random sampling draws a single random sample from the architecture space and returns it.

    is almost the same as random search thoo so maybe find out what is different

    """
    
    
    
    
    return 2

             
             

## Bayesian Optimisation

In [14]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, WhiteKernel


OPERATIONS = [
    "none",
    "skip_connect",
    "nor_conv_1x1",
    "nor_conv_3x3",
    "avg_pool_3x3"
]

def encode_structure(idx):
    arch_str = api.arch(idx)
    structure = []
    for dst, group in enumerate(arch_str.split('+'), start=1):
            items = group.strip().strip('|').split('|')
            for src, item in enumerate(items):
                structure.append((item.split('~')[0]))

    return structure


def UCB(mean, std, k):
    return mean + k * std

def bayesian_optimization(seed, hp_schedule, n_evals, dataset):
    # dataset init
    if dataset == 'cifar10':
        name = 'cifar10-valid'
    else:
        name = dataset

    records = []
    rng = np.random.default_rng(seed)
    best_val_accuracy = -float("inf")
    best_test_accuracy = 0
    best_idx = None
    cumulative_cost = 0.0

    
    encoder = OneHotEncoder(categories=[OPERATIONS] * 6, sparse_output=False) # categorie for each edge 
    encoder.fit([encode_structure(0)]) # init encoder

    # func to enocde the arcitecture
    def encode_architecture(idx):
        structure = encode_structure(idx)
        encoded = encoder.transform([structure])

        return encoded[0]


    #surgorate model init
    gp = GaussianProcessRegressor(
        kernel=RBF() + WhiteKernel(),
        normalize_y=True,
        random_state=seed
    )
    
    size = min(10, n_evals) # in case its more than the given value

    initial_indices = rng.choice(len(api), size=size, replace=False) # rn 10 but we can change this
    used_idx = set(int(idx) for idx in initial_indices)

    X = np.array([
         encode_architecture(int(idx))
         for idx in initial_indices
    ]) # features

    y = [] # result of valid accuray

    for i, indice in enumerate(initial_indices):
        info = api.get_more_info(int(indice), name, hp=hp_schedule, is_random=False)
        y.append(info["valid-accuracy"]) # needed for the algorithm

        # done for the results
        val_accuracy = info["valid-accuracy"]
        test_accuracy = info["test-accuracy"]
        if best_val_accuracy < val_accuracy:
            best_val_accuracy = val_accuracy
            best_test_accuracy = test_accuracy
            best_idx = int(indice)
        
        cumulative_cost += info["train-all-time"]
        records.append({
                "idx": int(indice),
                "best_idx": best_idx,
                "evaluation": i + 1,
                "val_accuracy": val_accuracy,
                "test_accuracy": test_accuracy,
                "best_val_accuracy": best_val_accuracy,
                "best_test_accuracy": best_test_accuracy,
                "training_cost": info["train-all-time"], # the cost of evaluating this particular architecture.
                "computation_cost": cumulative_cost,
                "epochs_trained": hp_schedule # amount of epochs trained
        })

    for i in range(n_evals - len(initial_indices)):
        gp.fit(X, y) # fitting on the surrogate model 

        # calc the rest of the indices
        candidate_indices = [
            idx for idx in range(len(api))
            if idx not in used_idx 
        ]

        X_candidates =  np.array([
            encode_architecture(idx)
            for idx in candidate_indices
        ])

        mean, std = gp.predict(X_candidates, return_std=True)
        ucb = UCB(mean, std, k=2)

        position = np.argmax(ucb) # need to be a full int
        candidate = candidate_indices[position]

        
        used_idx.add(candidate) 
        X = np.vstack([X , encode_architecture(candidate)])
        info = api.get_more_info(int(candidate), name, hp=hp_schedule, is_random=False)
        y.append(info["valid-accuracy"])

        val_accuracy = info["valid-accuracy"]
        test_accuracy = info["test-accuracy"]
        if best_val_accuracy < val_accuracy:
            best_val_accuracy = val_accuracy
            best_test_accuracy = test_accuracy
            best_idx = candidate
        
        cumulative_cost += info["train-all-time"]
        records.append({
                "idx": candidate,
                "best_idx": best_idx,
                "evaluation": size + i + 1, # 10 + i + 1
                "val_accuracy": val_accuracy,
                "test_accuracy": test_accuracy,
                "best_val_accuracy": best_val_accuracy,
                "best_test_accuracy": best_test_accuracy,
                "training_cost": info["train-all-time"], # the cost of evaluating this particular architecture.
                "computation_cost": cumulative_cost,
                "epochs_trained": hp_schedule, # amount of epochs trained
        })

    return records






# print(encode_structure(3))
# print(encode_architecture(3))
# print(encode_architecture(3).shape)

## Successive Halving

In [15]:
# start with n_evals
# remove worst half each time
def succesive_halving():

    return 2

## Iteration-based learning curve method 

In [ ]:
from scipy.optimize import curve_fit

# will implement the pow3 law (c-at^-a)

# Model: f(t) = c - a * t^(-alpha)
# t) training epoch
# f(t) predicited validation acccuracy
# c)  asymptot accuracy, the accuracy the curve approaches
# a) scale parameter, control how far acc is below c
# alpha) learning rate of the curve
# a, alpha and c are estimated by fitting the observed learning curve
# could use scipy.optimize.curve_fit


# -Partially train an architecture
# -store the learning curve
# =fit a parametric model
# -extraploate
# -use prediction to decide if promising

def pow3law(t, c, a, alpha):
    return c - a * t ** (-alpha)


def iterative_learning_curve(seed, hp_schedule, n_evals, dataset, step_size):
    if dataset == 'cifar10':
        name = 'cifar10-valid'
    else:
        name = dataset
    
    results = []
    records = []
    rng = np.random.default_rng(seed)
    used_idx = set()
    best_val_accuracy = -float("inf")
    best_test_accuracy = 0
    best_idx = None
    cumulative_cost = 0.0

    for i in range(n_evals):
        previous_training_cost = 0.0
        # 1, get a random architecture        
        idx = rng.integers(0, len(api))
        # we dont want repeated indexs
        while idx in used_idx:
            idx = rng.integers(0, len(api))

        used_idx.add(idx)

        # init first found idx
        if i == 0: 
            info = api.get_more_info(int(idx), name, hp=hp_schedule, is_random=False)

            # since first entry, no if loop
            best_val_accuracy = info["valid-accuracy"]
            best_test_accuracy = info["test-accuracy"]
            best_idx = int(idx)

            cumulative_cost += info["train-all-time"] - previous_training_cost
            records.append({
                        "idx": int(idx),
                        "best_idx": best_idx,
                        "evaluation": i + 1,
                        "val_accuracy": best_val_accuracy,
                        "predicted_val_accuracy": None,
                        "test_accuracy": best_test_accuracy,
                        "best_val_accuracy": best_val_accuracy,
                        "best_test_accuracy": best_test_accuracy,
                        "training_cost": info["train-all-time"], # the cost of evaluating this particular architecture.
                        "computation_cost": cumulative_cost,
                        "epochs_trained": hp_schedule, # amount of epochs trained
                    })
            continue
            

        # 2. train it for a certain amount of steps
        step_count = min(step_size, hp_schedule) # since we dont want more step than hp_schedule
        while step_count <= hp_schedule:

            # part when we reach the hp_schedule
            # we dont predict anymore, we get info at hp_schedule
            if step_count == hp_schedule:
                info = api.get_more_info(int(idx), name, hp=hp_schedule, is_random=False)
                val_accuracy = info["valid-accuracy"]
                test_accuracy = info["test-accuracy"]
                if best_val_accuracy < val_accuracy:
                    best_val_accuracy = val_accuracy
                    best_test_accuracy = test_accuracy
                    best_idx = int(idx)

                training_cost = info["train-all-time"] - previous_training_cost
                cumulative_cost += training_cost
                previous_training_cost = info["train-all-time"]


                records.append({
                            "idx": int(idx),
                            "best_idx": best_idx,
                            "evaluation": i + 1,
                            "val_accuracy": val_accuracy,
                            "predicted_val_accuracy": None,
                            "test_accuracy": test_accuracy,
                            "best_val_accuracy": best_val_accuracy,
                            "best_test_accuracy": best_test_accuracy,
                            "training_cost": training_cost, # the cost of evaluating this particular architecture.
                            "computation_cost": cumulative_cost,
                            "epochs_trained": hp_schedule, # amount of epochs trained
                        })
                break # stop the while loop



            # setting up the curve fit 
            X = [] # epoch
            y = [] # val acc 

            # filling up the data
            for epoch in range(step_count):
                results = api.get_more_info(int(idx), name, iepoch=epoch  ,hp=hp_schedule, is_random=False) # iepoch controlls at what step we end
                X.append(epoch + 1) # since we dont start at epoch 0
                y.append(results["valid-accuracy"])

            # might need to change for cases where curve_fit doesnt work
            params, _ = curve_fit(pow3law, 
                                  X, 
                                  y,
                                  p0=[max(y), max(y) - y[0] , 0.5], 
                                  maxfev=10000)

            # use params to get the calculated simulated_y
            simulated_y = pow3law(hp_schedule, *params)

            # Calculate additional training cost
            training_cost = results["train-all-time"] - previous_training_cost
            cumulative_cost += training_cost
            previous_training_cost = results["train-all-time"]

            val_accuracy = results["valid-accuracy"]

            # test score only done on the hp_schedule version        
            records.append({
                    "idx": int(idx),
                    "best_idx": best_idx,
                    "evaluation": i + 1, # 10 + i + 1
                    "val_accuracy": val_accuracy,
                    "predicted_val_accuracy": simulated_y,
                    "test_accuracy": None,
                    "best_val_accuracy": best_val_accuracy,
                    "best_test_accuracy": best_test_accuracy,
                    "training_cost": training_cost, # the cost of evaluating this particular architecture.
                    "computation_cost": cumulative_cost,
                    "epochs_trained": step_count, # amount of epochs trained
            })


            if simulated_y > best_val_accuracy:
                step_count = min(step_count + step_size, hp_schedule)
                continue
            else:
                break 

                # if it worse we drop it
                # but maybe still save info about this (like vall_acc that the current epoch has )


    return records


# Testing

In [17]:
# for now we do 3 seeds and 2 databases
DATASETS = ['cifar10', 'ImageNet16-120'] # different image sizes and data characterisitc
SEEDS = [20, 21, 22]
N_CONFIGS = 32 # amount of architectures evaluated # probabbly do a different one
HP_SCHEDULE = 200 # amount of epochs there is being trainted on

def experiment(function, **kwargs):
    all_results = {}

    for dataset in DATASETS:
        print(f"--- {dataset} ---")
        all_results[dataset] = {}

        for seed in SEEDS:
            print(f"--- {seed} ---")

            start = time.perf_counter()

            records = function(seed=seed,
                               hp_schedule=HP_SCHEDULE,
                               n_evals=N_CONFIGS,
                               dataset=dataset,
                               **kwargs
                               )

            runtime = time.perf_counter() - start 

            all_results[dataset][seed] = {
                "records": records,
                "runtime": runtime
            }

            final = records[-1]
            
            print(
                f"{dataset} | Seed {seed} | "
                f"Runtime: {runtime:.4f}s | "
                f"Records amount {len(records)} \n"
                f"Best idx: {final['best_idx']} | "
                f"Val: {final['best_val_accuracy']:.2f}% | "
                f"Test: {final['best_test_accuracy']:.2f}%"
            )

      
    # to have an idea how long the full experment was running
    total_runtime = sum(
        all_results[dataset][seed]["runtime"]
        for dataset in DATASETS
        for seed in SEEDS
    )
    print(f"\nTotal runtime: {total_runtime:.4f} seconds")
    print("#" * 42)

    return all_results


## running experiments

In [19]:
# os.makedirs("results", exist_ok=True)

# print(f"--- Random Search ---")
# rs_result = experiment(random_search)

# with open("results/random_search_results.pkl", "wb") as f:
#     pickle.dump(rs_result, f)

# print(f"--- Bayesian Optimization ---")
# ba_result = experiment(bayesian_optimization)

# with open("results/bayesian_optimization_results.pkl", "wb") as f:
#     pickle.dump(ba_result, f)

print(f"--- Iterative Learning Curve (Pow3) --- ")
step_size = 40 # a step count is necessary for this algorithm
ilc_result = experiment(iterative_learning_curve, step_size=step_size)
# some minor checks how it performed
full_records = [
    record for record in ilc_result
    if record["epochs_trained"] == 200
]


print(f"Total records: {len(ilc_result)}")
print(f"Unique architectures: {len(set(r['idx'] for r in ilc_result))}")
print(f"Fully evaluated architectures: {len(full_records)}")



--- Iterative Learning Curve (Pow3) --- 
--- cifar10 ---
--- 20 ---


C:\Users\kayad\AppData\Local\Temp\ipykernel_17796\4252670468.py:124: OptimizeWarning: Covariance of the parameters could not be estimated
  params, _ = curve_fit(pow3law, X, y)
C:\Users\kayad\AppData\Local\Temp\ipykernel_17796\4252670468.py:22: RuntimeWarning: overflow encountered in power
  return c - a * t ** (-alpha)


RuntimeError: Optimal parameters not found: Number of calls to function has reached maxfev = 800.

In [ ]:
for epoch in [19, 39, 59, 199]:
    info = api.get_more_info(
        0,
        "cifar10-valid",
        iepoch=epoch,
        hp=200,
        is_random=False
    )

    print(epoch + 1, info["train-all-time"])

20 144.42185997962952
40 288.84371995925903
60 433.26557993888855
200 1444.2185997962952
